In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dropout,Dense,Input,Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])

In [6]:
def clean(text):
    text=str.lower(text)
    text=re.sub("[<br>/>]"," ",text)
    text=re.sub("[^a-zA-Z0-9]"," ",text)
    return text

In [7]:
df['review']=df['review'].apply(clean)

In [8]:
df.head()

,review,sentiment
0,one of the othe eviewe s has mentioned that ...,1
1,a wonde ful little p oduction the...,1
2,i thought this was a wonde ful way to spend ti...,1
3,asically the e s a family whe e a little oy ...,0
4,pette mattei s love in the time of money is...,1


In [9]:
X=df['review']
y=df['sentiment']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)


In [10]:
X_train.shape

(40000,)

In [11]:
tokenizer=Tokenizer(num_words=1000)
tokenizer.fit_on_texts(X_train)
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)


In [12]:
X_train_pad=pad_sequences(
    X_train_seq,
    maxlen=200,
    padding='post',
    truncating='post'
)
X_test_pad=pad_sequences(
    X_test_seq,
    maxlen=200,
    padding='post',
    truncating='post'
)

In [13]:
model=Sequential([
    Input(shape=(200,)),
    Embedding(1000,64),
    LSTM(128),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [15]:
model.fit(X_train_pad,y_train,epochs=2,validation_split=0.2,batch_size=32)

Epoch 1/2
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 225s 222ms/step - accuracy: 0.5295 - loss: 0.6872 - val_accuracy: 0.5779 - val_loss: 0.7122
Epoch 2/2
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 239s 199ms/step - accuracy: 0.5738 - loss: 0.6698 - val_accuracy: 0.5798 - val_loss: 0.6812


In [16]:
loss,accuaracy=model.evaluate(X_test_pad,y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 97ms/step - accuracy: 0.5827 - loss: 0.6763


In [17]:
pred=model.predict(X_test_pad[7:8])
if(pred>0.5):
    print("prediction :positive")
else:
    print("prediction: negative")
actual=y_test[7]
if(actual>=0.5):
    print("Actual: positive")
else:
    print("Actual:negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
prediction: negative
Actual:negative
